# 10 - Crawl LangChain Documentation with Tavily

This notebook isolates the crawling portion of the instructor's `ingestion.py`.

```text
LangChain documentation URL
→ TavilyCrawl
→ LangChain Document objects
→ save documents for notebook 11
```

The only notebook-specific addition is saving the crawled documents locally so the Tavily crawl does not need to be repeated before ingestion.

## 1. Optional installation

In [ ]:
# %pip install -U langchain-tavily python-dotenv truststore

## 2. SSL, imports, and environment variables

In [3]:
import truststore
truststore.inject_into_ssl()

from pathlib import Path
ENV_PATH = Path("../langchain-course/.env").resolve()

from dotenv import load_dotenv
load_dotenv(dotenv_path=ENV_PATH)
print("TAVILY_API_KEY loaded:", bool(os.getenv("TAVILY_API_KEY")))


import json
import os
from langchain_core.documents import Document
from langchain_tavily import TavilyCrawl

TAVILY_API_KEY loaded: True


## 3. Configuration

The instructor uses Tavily advanced extraction and a crawl depth of 2. The current LangChain documentation URL is used in place of the older documentation URL.

In [5]:
START_URL = "https://docs.langchain.com/oss/python/langchain/overview"
OUTPUT_FILE = Path("langchain_docs.json")


## 4. Crawl the documentation

This cell calls Tavily and consumes Tavily credits.

In [6]:
tavily_crawl = TavilyCrawl(
    max_depth=2,
    extract_depth="advanced",
)

crawl_results = tavily_crawl.invoke({
    "url": START_URL,
})

print("Pages returned:", len(crawl_results["results"]))


Pages returned: 42


In [8]:
crawl_results.keys()

dict_keys(['base_url', 'results', 'response_time', 'request_id'])

In [11]:
len(crawl_results['results'])

42

In [10]:
crawl_results['results'][0]

{'url': 'https://docs.langchain.com/oss/python/langchain/overview',
 'raw_content': '## Documentation Index\n\nFetch the complete documentation index at: </llms.txt>\n\nUse this file to discover all available pages before exploring further.\n\nFall AMA Series: Live sessions on building, evaluating, deploying, and continously improving agents with LangSmith. [Register now →](https://events.langchain.com/fall-product-series/)\n\n![light logo](https://mintcdn.com/langchain-5e9cc07a/nQm-sjd_MByLhgeW/images/brand/langchain-docs-dark-blue.png?fit=max&auto=format&n=nQm-sjd_MByLhgeW&q=85&s=5babf1a1962208fd7eed942fa2432ecb)\n![dark logo](https://mintcdn.com/langchain-5e9cc07a/nQm-sjd_MByLhgeW/images/brand/langchain-docs-light-blue.png?fit=max&auto=format&n=nQm-sjd_MByLhgeW&q=85&s=0bcd2a1f2599ed228bcedf0f535b45b1)\n\nPython\n\n### Get started\n\n### Core components\n\n### Middleware\n\n### Frontend\n\n### Advanced usage\n\n### Agent development\n\n### Production\n\n## On this page\n\n# LangChain

## 5. Convert crawl results to LangChain Documents

This matches the instructor's core conversion: page content becomes `page_content`, and the original URL becomes `metadata["source"]`.

In [12]:
documents = [
    Document(
        page_content=result["raw_content"],
        metadata={"source": result["url"]},
    )
    for result in crawl_results["results"]
    if result.get("raw_content")
]

print("Documents created:", len(documents))
print("First source:", documents[0].metadata["source"])
print(documents[0].page_content[:1000])


Documents created: 40
First source: https://docs.langchain.com/oss/python/langchain/overview
## Documentation Index

Fetch the complete documentation index at: </llms.txt>

Use this file to discover all available pages before exploring further.

Fall AMA Series: Live sessions on building, evaluating, deploying, and continously improving agents with LangSmith. [Register now →](https://events.langchain.com/fall-product-series/)

![light logo](https://mintcdn.com/langchain-5e9cc07a/nQm-sjd_MByLhgeW/images/brand/langchain-docs-dark-blue.png?fit=max&auto=format&n=nQm-sjd_MByLhgeW&q=85&s=5babf1a1962208fd7eed942fa2432ecb)
![dark logo](https://mintcdn.com/langchain-5e9cc07a/nQm-sjd_MByLhgeW/images/brand/langchain-docs-light-blue.png?fit=max&auto=format&n=nQm-sjd_MByLhgeW&q=85&s=0bcd2a1f2599ed228bcedf0f535b45b1)

Python

### Get started

### Core components

### Middleware

### Frontend

### Advanced usage

### Agent development

### Production

## On this page

# LangChain overview

LangChain 

## 6. Save documents for notebook 11

Saving is required only because the instructor's single `ingestion.py` script has been split into separate course notebooks.

In [13]:
OUTPUT_FILE.write_text(
    json.dumps(
        [
            {
                "page_content": document.page_content,
                "metadata": document.metadata,
            }
            for document in documents
        ],
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("Saved:", OUTPUT_FILE.resolve())


Saved: C:\Users\inkk1r04\OneDrive - Kellogg Company\Desktop\Work\0. DS Capability\Courses\Langchain\langchain-course-jupyter\langchain_docs.json
